# Rodada 3 — tendências, choques de bioma e janela temporal

Sensibilidades que podem absorver tanto confundimento quanto o próprio mecanismo; não são testes definitivos de causalidade.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0,str(Path.cwd().parent/'testes2'))
from metodos import *
from IPython.display import display
e=Experimento(Path.cwd());d=e.d
# O CSV já traz preço t−1; shifts de 1 e 2 criam t−2 e t−3.
for cultura in ['soja','milho']:
    for lag in [1,2,3]:
        d[f'expo_{cultura}_l{lag}']=d.groupby('id')['expo_'+cultura].shift(lag-1).where(d.ano.eq(d.groupby('id').ano.shift(lag-1)+lag-1))
lagcols=[f'expo_{c}_l{l}' for c in ['soja','milho'] for l in [1,2,3]]

## 1. Tendências lineares municipais — dois destinos

In [2]:
s=d[d.ano.ge(2011)].copy() if False else d.copy()
for u in ['agricultura','pastagem']:
    r=e.linear('B03_'+u,'pp_'+u,['expo_soja','expo_milho'],s,familia='robustez_temporal',trend=True,bioma_ano=False)
    display(pd.DataFrame(e.coef).tail(2)[['id','termo','coef','ic95_inf','ic95_sup','N','anos','condicao']])

,id,termo,coef,ic95_inf,ic95_sup,N,anos,condicao
0,B03_agricultura,expo_soja,0.018265,-0.003094,0.039624,2772,24,1.911826
1,B03_agricultura,expo_milho,-0.002316,-0.009727,0.005096,2772,24,1.911826


,id,termo,coef,ic95_inf,ic95_sup,N,anos,condicao
2,B03_pastagem,expo_soja,0.086359,0.024434,0.148283,2772,24,1.911826
3,B03_pastagem,expo_milho,0.045683,0.004433,0.086933,2772,24,1.911826


## 2. FE bioma × ano — dois destinos

In [3]:
s=d[d.ano.ge(2011)].copy() if False else d.copy()
for u in ['agricultura','pastagem']:
    r=e.linear('B04_'+u,'pp_'+u,['expo_soja','expo_milho'],s,familia='robustez_temporal',trend=False,bioma_ano=True)
    display(pd.DataFrame(e.coef).tail(2)[['id','termo','coef','ic95_inf','ic95_sup','N','anos','condicao']])

,id,termo,coef,ic95_inf,ic95_sup,N,anos,condicao
4,B04_agricultura,expo_soja,0.028211,-0.000950,0.057373,2772,24,1.927982
5,B04_agricultura,expo_milho,0.009159,-0.007197,0.025516,2772,24,1.927982


,id,termo,coef,ic95_inf,ic95_sup,N,anos,condicao
6,B04_pastagem,expo_soja,0.017760,-0.052984,0.088504,2772,24,1.927982
7,B04_pastagem,expo_milho,0.123683,0.017889,0.229478,2772,24,1.927982


## 3. Somente 2011–2024 — dois destinos

In [4]:
s=d[d.ano.ge(2011)].copy() if True else d.copy()
for u in ['agricultura','pastagem']:
    r=e.linear('B05_'+u,'pp_'+u,['expo_soja','expo_milho'],s,familia='robustez_temporal',trend=False,bioma_ano=False)
    display(pd.DataFrame(e.coef).tail(2)[['id','termo','coef','ic95_inf','ic95_sup','N','anos','condicao']])

,id,termo,coef,ic95_inf,ic95_sup,N,anos,condicao
8,B05_agricultura,expo_soja,-0.004754,-0.015997,0.006490,1635,14,1.925776
9,B05_agricultura,expo_milho,-0.002594,-0.006904,0.001715,1635,14,1.925776


,id,termo,coef,ic95_inf,ic95_sup,N,anos,condicao
10,B05_pastagem,expo_soja,0.040440,-0.069683,0.150564,1635,14,1.925776
11,B05_pastagem,expo_milho,0.061267,-0.013238,0.135772,1635,14,1.925776


## 4. Mudanças anuais — primeiras diferenças

Diferença de taxa, não taxa de crescimento. Somente pares anuais consecutivos. FE municipal nas diferenças controla mudança média municipal.

In [5]:
s=d.copy()
for col in ['pp_agricultura','pp_pastagem','expo_soja','expo_milho']:
    s['delta_'+col]=s.groupby('id')[col].diff().where(s.ano.eq(s.groupby('id').ano.shift(1)+1))
s=s.dropna(subset=['delta_pp_agricultura','delta_pp_pastagem','delta_expo_soja','delta_expo_milho'])
for u in ['agricultura','pastagem']:
    r=e.linear('B06_'+u,'delta_pp_'+u,['delta_expo_soja','delta_expo_milho'],s,familia='robustez_temporal')
    display(pd.DataFrame(e.coef).tail(2)[['id','termo','coef','ic95_inf','ic95_sup','N']])

,id,termo,coef,ic95_inf,ic95_sup,N
12,B06_agricultura,delta_expo_soja,-0.002630,-0.023434,0.018173,2595
13,B06_agricultura,delta_expo_milho,-0.007729,-0.023202,0.007744,2595


,id,termo,coef,ic95_inf,ic95_sup,N
14,B06_pastagem,delta_expo_soja,-0.027453,-0.139160,0.084253,2595
15,B06_pastagem,delta_expo_milho,0.075727,-0.004046,0.155501,2595


## 5. Diferenças formais entre destinos sob controles ampliados

In [6]:
for ident,trend,ba in [('D03_diferenca',True,False),('D04_diferenca',False,True)]:
    e.linear(ident,'diferenca_pp',['expo_soja','expo_milho'],familia='diferencas',trend=trend,bioma_ano=ba)
c=pd.DataFrame(e.coef).set_index(['id','termo'])
for b,dd in [('B03','D03'),('B04','D04')]:
    for term in ['expo_soja','expo_milho']:assert np.isclose(c.loc[(dd+'_diferenca',term),'coef'],c.loc[(b+'_pastagem',term),'coef']-c.loc[(b+'_agricultura',term),'coef'],atol=1e-8)
e.salvar('tendencias')

## 6. Quanto da variação resta depois de cada absorção?

In [7]:
s=d.dropna(subset=['pp_agricultura','pp_pastagem','expo_soja','expo_milho'])
base,_,_=projetar(s,['expo_soja','expo_milho'])
rows=[]
for nome,trend,ba in [('municipio+ano',False,False),('tendencias',True,False),('bioma×ano',False,True)]:
    v,rank,it=projetar(s,['expo_soja','expo_milho'],trend,ba)
    for j,col in enumerate(['expo_soja','expo_milho']):rows.append({'controle':nome,'variavel':col,'variancia_residual':v[:,j].var(),'fracao_variancia_vs_TWFE':v[:,j].var()/base[:,j].var(),'rank_FE':rank,'iteracoes':it})
r=pd.DataFrame(rows);r.to_csv(e.p/'auditoria_variacao_residual.csv',index=False);display(r)

,controle,variavel,variancia_residual,fracao_variancia_vs_TWFE,rank_FE,iteracoes
0,municipio+ano,expo_soja,1.048039,1.000000,160,11
1,municipio+ano,expo_milho,0.728633,1.000000,160,11
2,tendencias,expo_soja,0.839512,0.801031,293,11
3,tendencias,expo_milho,0.630240,0.864963,293,11
4,bioma×ano,expo_soja,0.702734,0.670522,206,20
5,bioma×ano,expo_milho,0.445616,0.611579,206,20
